In [ ]:
!pip install -q transformers torch gensim
!pip install -q nltk

In [ ]:
import numpy as np
import pandas as pd
import nltk
nltk.download('punkt')

from sklearn.model_selection import train_test_split
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Embedding, SimpleRNN, LSTM, GRU, Dense
from nltk.translate.bleu_score import corpus_bleu, SmoothingFunction

from gensim.models import FastText

import torch
from transformers import AutoTokenizer, AutoModel


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


In [ ]:
data = pd.read_csv("NLP_QA_MT.csv")
df = data.dropna(subset=['Answer_QA']).copy()

def safe_join(x):
    if isinstance(x, list):
        return ' '.join(x)
    return str(x)

df['input_text']  = df['QuestionText_QA'].apply(safe_join)
df['target_text'] = df['Answer_QA'].apply(safe_join)
df['target_text'] = df['target_text'].apply(lambda x: 'start ' + x + ' end')

X_train, X_test, y_train, y_test = train_test_split(
    df['input_text'], df['target_text'],
    test_size=0.2, random_state=42, shuffle=True
)

print("Training data:", X_train.shape)
print("Testing data:", X_test.shape)
print(X_train.iloc[0])
print(y_train.iloc[0])

Training data: (3716,)
Testing data: (929,)
['اذا', 'تعن', 'لك', 'وسق']
start ['وسق', 'تعن', 'لي', 'وسل', 'عبر', 'عن', 'شعر', 'رخء', 'ماع'] end


In [ ]:
#Tokenizers
input_tokenizer = Tokenizer(num_words=5000)
input_tokenizer.fit_on_texts(X_train)

target_tokenizer = Tokenizer()
target_tokenizer.fit_on_texts(y_train)

#Sequences
X_train_seq = input_tokenizer.texts_to_sequences(X_train)
X_test_seq  = input_tokenizer.texts_to_sequences(X_test)

Y_train_seq = target_tokenizer.texts_to_sequences(y_train)
Y_test_seq  = target_tokenizer.texts_to_sequences(y_test)

#Padding
max_encoder_len = 150
max_decoder_len = 50

X_train_pad = pad_sequences(X_train_seq, maxlen=max_encoder_len, padding='post')
X_test_pad  = pad_sequences(X_test_seq,  maxlen=max_encoder_len, padding='post')

Y_train_pad = pad_sequences(Y_train_seq, maxlen=max_decoder_len, padding='post', truncating='post')
Y_test_pad  = pad_sequences(Y_test_seq,  maxlen=max_decoder_len, padding='post', truncating='post')

#Decoder inputs/targets
decoder_input_data  = Y_train_pad[:, :-1]
decoder_target_data = Y_train_pad[:, 1:]
decoder_target_data = decoder_target_data.reshape(
    decoder_target_data.shape[0], decoder_target_data.shape[1], 1
)

#Vocab sizes
encoder_vocab_size = len(input_tokenizer.word_index) + 1
decoder_vocab_size = len(target_tokenizer.word_index) + 1

EMBEDDING_DIM = 128
HIDDEN_DIM    = 256
EPOCHS        = 8
BATCH_SIZE    = 64

print("Encoder vocab:", encoder_vocab_size)
print("Decoder vocab:", decoder_vocab_size)
print("Encoder shape:", X_train_pad.shape)
print("Decoder input shape:", decoder_input_data.shape)
print("Decoder target shape:", decoder_target_data.shape)

Encoder vocab: 2375
Decoder vocab: 4083
Encoder shape: (3716, 150)
Decoder input shape: (3716, 49)
Decoder target shape: (3716, 49, 1)


In [ ]:
def build_seq2seq_external_emb(cell_type,
                                encoder_emb_matrix,
                                decoder_emb_matrix,
                                max_enc_len, max_dec_len,
                                embedding_dim, hidden_dim,
                                decoder_vocab_size):
    #Encoder
    encoder_inputs = Input(shape=(max_enc_len,))
    enc_emb = Embedding(
        input_dim=encoder_emb_matrix.shape[0],
        output_dim=encoder_emb_matrix.shape[1],
        weights=[encoder_emb_matrix],
        trainable=False,
        mask_zero=True
    )(encoder_inputs)

    if cell_type == 'rnn':
        _, enc_state = SimpleRNN(hidden_dim, return_state=True)(enc_emb)
        encoder_states = [enc_state]

    elif cell_type == 'lstm':
        _, enc_h, enc_c = LSTM(hidden_dim, return_state=True)(enc_emb)
        encoder_states = [enc_h, enc_c]

    elif cell_type == 'gru':
        _, enc_state = GRU(hidden_dim, return_state=True)(enc_emb)
        encoder_states = [enc_state]

    #Decoder
    decoder_inputs = Input(shape=(max_dec_len - 1,))
    dec_emb_layer = Embedding(
        input_dim=decoder_emb_matrix.shape[0],
        output_dim=decoder_emb_matrix.shape[1],
        weights=[decoder_emb_matrix],
        trainable=False,
        mask_zero=True
    )
    dec_emb = dec_emb_layer(decoder_inputs)

    if cell_type == 'rnn':
        decoder_rnn = SimpleRNN(hidden_dim, return_sequences=True, return_state=True)
        decoder_outputs, _ = decoder_rnn(dec_emb, initial_state=encoder_states[0])

    elif cell_type == 'lstm':
        decoder_lstm = LSTM(hidden_dim, return_sequences=True, return_state=True)
        decoder_outputs, _, _ = decoder_lstm(dec_emb, initial_state=encoder_states)

    elif cell_type == 'gru':
        decoder_gru = GRU(hidden_dim, return_sequences=True, return_state=True)
        decoder_outputs, _ = decoder_gru(dec_emb, initial_state=encoder_states[0])

    decoder_dense   = Dense(decoder_vocab_size, activation='softmax')
    decoder_outputs = decoder_dense(decoder_outputs)

    model = Model([encoder_inputs, decoder_inputs], decoder_outputs)
    model.compile(optimizer='adam',
                  loss='sparse_categorical_crossentropy',
                  metrics=['accuracy'])

    return model, dec_emb_layer, decoder_dense


In [ ]:
def build_inference_models(cell_type, encoder_inputs, enc_emb,
                            dec_emb_layer, decoder_dense,
                            hidden_dim, max_dec_len,
                            encoder_states_tensors):
    #Encoder inference
    encoder_model = Model(encoder_inputs, encoder_states_tensors
                          if isinstance(encoder_states_tensors, list)
                          else [encoder_states_tensors])

    #Decoder inference
    decoder_inputs_inf = Input(shape=(1,))
    dec_emb_inf = dec_emb_layer(decoder_inputs_inf)

    if cell_type in ('rnn', 'gru'):
        state_input = Input(shape=(hidden_dim,))
        CellLayer = SimpleRNN if cell_type == 'rnn' else GRU

        if cell_type == 'rnn':
            cell_inf = SimpleRNN(hidden_dim, return_sequences=True, return_state=True)
        else:
            cell_inf = GRU(hidden_dim, return_sequences=True, return_state=True)
        out_inf, state_out = cell_inf(dec_emb_inf, initial_state=state_input)
        out_inf = decoder_dense(out_inf)
        decoder_model = Model([decoder_inputs_inf, state_input],
                               [out_inf, state_out])

    elif cell_type == 'lstm':
        state_h_input = Input(shape=(hidden_dim,))
        state_c_input = Input(shape=(hidden_dim,))
        cell_inf = LSTM(hidden_dim, return_sequences=True, return_state=True)
        out_inf, h_out, c_out = cell_inf(dec_emb_inf,
                                          initial_state=[state_h_input, state_c_input])
        out_inf = decoder_dense(out_inf)
        decoder_model = Model([decoder_inputs_inf, state_h_input, state_c_input],
                               [out_inf, h_out, c_out])

    return encoder_model, decoder_model


In [ ]:
def decode_sequence(input_seq, encoder_model, decoder_model,
                    cell_type, max_dec_len, tgt_tokenizer=None):

    tok = tgt_tokenizer if tgt_tokenizer is not None else target_tokenizer

    states = encoder_model.predict(input_seq, verbose=0)
    if cell_type == 'lstm':
        state_h, state_c = states[0], states[1]
    else:
        state_val = states[0]

    target_seq = np.zeros((1, 1))
    target_seq[0, 0] = tok.word_index.get('start', 1)

    decoded = []
    prev_word = None

    for _ in range(max_dec_len):
        if cell_type == 'lstm':
            output_tokens, state_h, state_c = decoder_model.predict(
                [target_seq, state_h, state_c], verbose=0
            )
        else:
            output_tokens, state_val = decoder_model.predict(
                [target_seq, state_val], verbose=0
            )

        sampled_index = np.argmax(output_tokens[0, -1, :])
        word = tok.index_word.get(sampled_index, '')

        if word in ['end', '', None]:
            break
        if word == prev_word:      # repetition loop → stop
            break

        decoded.append(word)
        prev_word = word
        target_seq[0, 0] = sampled_index

    return ' '.join(decoded)

In [ ]:
def evaluate_bleu(encoder_model, decoder_model, cell_type, label,
                  test_pad=None, tgt_tokenizer=None):

    _test_pad      = test_pad      if test_pad      is not None else X_test_pad
    _tgt_tokenizer = tgt_tokenizer if tgt_tokenizer is not None else target_tokenizer

    eval_samples = min(100, len(_test_pad))  #use 100 samples for BLEU evaluation

    predictions = []
    for i in range(eval_samples):
        seq  = _test_pad[i].reshape(1, max_encoder_len)
        pred = decode_sequence(seq, encoder_model, decoder_model,
                       cell_type, max_decoder_len, tgt_tokenizer=_tgt_tokenizer)
        predictions.append(pred)

    references  = [text.lower().strip() for text in y_test[:eval_samples]]
    predictions = [text.lower().strip() for text in predictions]

    smooth = SmoothingFunction().method1
    references_tok  = [[r.split()] for r in references]
    predictions_tok = [p.split() for p in predictions]

    bleu = corpus_bleu(references_tok, predictions_tok,
                       smoothing_function=smooth)

    print(f"\n{'='*40}")
    print(f"BLEU Score [{label}]: {bleu:.4f}")
    print(f"{'='*40}")

    return bleu

all_scores = {}


# **One Hot Encoding**

In [ ]:
#One Hot Encodding
#used manageable size (3000x3000 = 36MB).

oh_input_tokenizer = Tokenizer(num_words=3000, oov_token='<OOV>')
oh_input_tokenizer.fit_on_texts(X_train)

oh_target_tokenizer = Tokenizer(num_words=3000, oov_token='<OOV>')
oh_target_tokenizer.fit_on_texts(y_train)

#Sequences
X_train_seq_oh = oh_input_tokenizer.texts_to_sequences(X_train)
X_test_seq_oh  = oh_input_tokenizer.texts_to_sequences(X_test)

Y_train_seq_oh = oh_target_tokenizer.texts_to_sequences(y_train)
Y_test_seq_oh  = oh_target_tokenizer.texts_to_sequences(y_test)

#Padding
X_train_pad_oh = pad_sequences(X_train_seq_oh, maxlen=max_encoder_len, padding='post')
X_test_pad_oh  = pad_sequences(X_test_seq_oh,  maxlen=max_encoder_len, padding='post')

Y_train_pad_oh = pad_sequences(Y_train_seq_oh, maxlen=max_decoder_len,
                                padding='post', truncating='post')

#Decoder inputs/targets
decoder_input_data_oh  = Y_train_pad_oh[:, :-1]
decoder_target_data_oh = Y_train_pad_oh[:, 1:]
decoder_target_data_oh = decoder_target_data_oh.reshape(
    decoder_target_data_oh.shape[0],
    decoder_target_data_oh.shape[1],
    1
)

#Vocab sizes (3000)
oh_encoder_vocab_size = min(len(oh_input_tokenizer.word_index) + 1, 3000)
oh_decoder_vocab_size = min(len(oh_target_tokenizer.word_index) + 1, 3000)

print(f"One-Hot encoder vocab size: {oh_encoder_vocab_size}")
print(f"One-Hot decoder vocab size: {oh_decoder_vocab_size}")

#Identity matrices
encoder_onehot_matrix = np.eye(oh_encoder_vocab_size, dtype='float32')
decoder_onehot_matrix = np.eye(oh_decoder_vocab_size, dtype='float32')

print(f"Encoder One-Hot matrix shape: {encoder_onehot_matrix.shape}")
print(f"Decoder One-Hot matrix shape: {decoder_onehot_matrix.shape}")
print(f"Memory estimate: ~{round(oh_encoder_vocab_size**2 * 4 / 1e6, 1)} MB per matrix")

One-Hot encoder vocab size: 2376
One-Hot decoder vocab size: 3000
Encoder One-Hot matrix shape: (2376, 2376)
Decoder One-Hot matrix shape: (3000, 3000)
Memory estimate: ~22.6 MB per matrix


In [ ]:
print("One-Hot + RNN")

model_oh_rnn, dec_emb_layer_oh_rnn, dec_dense_oh_rnn = build_seq2seq_external_emb(
    cell_type='rnn',
    encoder_emb_matrix=encoder_onehot_matrix,
    decoder_emb_matrix=decoder_onehot_matrix,
    max_enc_len=max_encoder_len,
    max_dec_len=max_decoder_len,
    embedding_dim=oh_encoder_vocab_size,
    hidden_dim=HIDDEN_DIM,
    decoder_vocab_size=oh_decoder_vocab_size
)
model_oh_rnn.summary()

model_oh_rnn.fit(
    [X_train_pad_oh, decoder_input_data_oh],
    decoder_target_data_oh,
    batch_size=BATCH_SIZE,
    epochs=EPOCHS,
    validation_split=0.2
)

One-Hot + RNN


Model: "functional_8"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_12      │ (None, 150)       │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ input_layer_13      │ (None, 49)        │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_6         │ (None, 150, 2376) │  5,645,376 │ input_layer_12[0… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ not_equal_8         │ (None, 150)       │          0 │ input_layer_12[0… │
│ (NotEqual)          │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_7         │ (None, 49, 3000)  │  9,000,000 │ input_layer_13[0… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ simple_rnn_8        │ [(None, 256),     │    674,048 │ embedding_6[0][0… │
│ (SimpleRNN)         │ (None, 256)]      │            │ not_equal_8[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ simple_rnn_9        │ [(None, 49, 256), │    833,792 │ embedding_7[0][0… │
│ (SimpleRNN)         │ (None, 256)]      │            │ simple_rnn_8[0][… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_2 (Dense)     │ (None, 49, 3000)  │    771,000 │ simple_rnn_9[0][… │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 16,924,216 (64.56 MB)

 Trainable params: 2,278,840 (8.69 MB)

 Non-trainable params: 14,645,376 (55.87 MB)

Epoch 1/8
47/47 ━━━━━━━━━━━━━━━━━━━━ 23s 293ms/step - accuracy: 0.0524 - loss: 6.7425 - val_accuracy: 0.1042 - val_loss: 6.3738
Epoch 2/8
47/47 ━━━━━━━━━━━━━━━━━━━━ 5s 78ms/step - accuracy: 0.0788 - loss: 6.3179 - val_accuracy: 0.1126 - val_loss: 6.2972
Epoch 3/8
47/47 ━━━━━━━━━━━━━━━━━━━━ 7s 127ms/step - accuracy: 0.0820 - loss: 6.2336 - val_accuracy: 0.0926 - val_loss: 6.2455
Epoch 4/8
47/47 ━━━━━━━━━━━━━━━━━━━━ 7s 67ms/step - accuracy: 0.0927 - loss: 6.1676 - val_accuracy: 0.1140 - val_loss: 6.2018
Epoch 5/8
47/47 ━━━━━━━━━━━━━━━━━━━━ 3s 68ms/step - accuracy: 0.1150 - loss: 6.0876 - val_accuracy: 0.1185 - val_loss: 6.1226
Epoch 6/8
47/47 ━━━━━━━━━━━━━━━━━━━━ 3s 67ms/step - accuracy: 0.1239 - loss: 5.9772 - val_accuracy: 0.1214 - val_loss: 6.0416
Epoch 7/8
47/47 ━━━━━━━━━━━━━━━━━━━━ 3s 66ms/step - accuracy: 0.1306 - loss: 5.8791 - val_accuracy: 0.1264 - val_loss: 6.0479
Epoch 8/8
47/47 ━━━━━━━━━━━━━━━━━━━━ 3s 66ms/step - accuracy: 0.1373 - loss: 5.8077 - val_accuracy: 0.1369 - val_lo

In [ ]:
#Inference and Evaluate
enc_inp_oh_rnn = model_oh_rnn.input[0]
enc_emb_oh_rnn = model_oh_rnn.layers[2].output

encoder_model_oh_rnn, decoder_model_oh_rnn = build_inference_models(
    cell_type='rnn',
    encoder_inputs=enc_inp_oh_rnn,
    enc_emb=enc_emb_oh_rnn,
    dec_emb_layer=dec_emb_layer_oh_rnn,
    decoder_dense=dec_dense_oh_rnn,
    hidden_dim=HIDDEN_DIM,
    max_dec_len=max_decoder_len,
    encoder_states_tensors=model_oh_rnn.get_layer(index=4).output
)

print("\nSample: One-Hot + RNN")
for i in range(3):
    print("\n====================")
    print("INPUT:", X_test.iloc[i])
    seq = X_test_pad_oh[i].reshape(1, max_encoder_len)
    print("PREDICTED:", decode_sequence(seq, encoder_model_oh_rnn,
                                        decoder_model_oh_rnn, 'rnn', max_decoder_len))

all_scores['OneHot_RNN'] = evaluate_bleu(
    encoder_model_oh_rnn, decoder_model_oh_rnn,
    'rnn', 'One-Hot + RNN',
    X_test_pad_oh, oh_target_tokenizer
)


Sample: One-Hot + RNN

INPUT: ['عرف', 'ادب']
PREDICTED: 'اشد' 'سنغافوري' 'حمراء' 'حيه' 'طبب' 'بدا' 'شكل' 'كزت' 'فيه' 'دور'

INPUT: ['عم', 'فكر', 'عند', 'قرء']
PREDICTED: '905' 'حل' 'بادل' 'زمل' 'يقض' 'كريستوفر' 'اذا' 'شخص' 'نخي' '21' 'يعب' 'قلل' 'عبدالناصر' 'ومايكروسوفت' 'درت' 'في' 'هرج' 'احد' '54' 'ساح' 'قلب' 'فنن' 'عرب' 'وخذ' 'واستهلاك' '2007' 'فيه' 'مثل' 'شيع' 'نمو' 'وقع' 'دسر' 'كنه' 'تبع'

INPUT: ['لمذ', 'ضرب', 'ضرب', 'رقم', 'صفر', 'يسو', 'وحد']
PREDICTED: 'رشح' 'سنغافوري' 'سنن' 'ناس' 'حتق' '415000' 'ركش' '1989' 'اله' 'بحر' 'نظم' 'عصر' 'علي' 'طعه' 'ثمن' 'كزت' 'يضع' 'ضطرار' 'زيد' 'يزه' 'تعل' 'ثقف' 'نفت' 'ثلج' 'مثل' 'كسد' 'كفح' 'نجه' 'هو' 'عيد' '1481م' 'ناخ' 'وظف' 'وخذ' 'مصر' 'سبب' 'رشد' 'سنو' 'حلي' 'غمض' 'الكترونيه' 'سكي' 'مع' 'من' 'عليه' 'عدد' 'عمر' 'مذج'

BLEU Score [One-Hot + RNN]: 0.0001


In [ ]:
print("One-Hot + LSTM")

model_oh_lstm, dec_emb_layer_oh_lstm, dec_dense_oh_lstm = build_seq2seq_external_emb(
    cell_type='lstm',
    encoder_emb_matrix=encoder_onehot_matrix,
    decoder_emb_matrix=decoder_onehot_matrix,
    max_enc_len=max_encoder_len,
    max_dec_len=max_decoder_len,
    embedding_dim=oh_encoder_vocab_size,
    hidden_dim=HIDDEN_DIM,
    decoder_vocab_size=oh_decoder_vocab_size
)
model_oh_lstm.summary()

model_oh_lstm.fit(
    [X_train_pad_oh, decoder_input_data_oh],
    decoder_target_data_oh,
    batch_size=BATCH_SIZE,
    epochs=EPOCHS,
    validation_split=0.2
)

One-Hot + LSTM


Model: "functional_11"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_16      │ (None, 150)       │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ input_layer_17      │ (None, 49)        │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_8         │ (None, 150, 2376) │  5,645,376 │ input_layer_16[0… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ not_equal_11        │ (None, 150)       │          0 │ input_layer_16[0… │
│ (NotEqual)          │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_9         │ (None, 49, 3000)  │  9,000,000 │ input_layer_17[0… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm (LSTM)         │ [(None, 256),     │  2,696,192 │ embedding_8[0][0… │
│                     │ (None, 256),      │            │ not_equal_11[0][… │
│                     │ (None, 256)]      │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm_1 (LSTM)       │ [(None, 49, 256), │  3,335,168 │ embedding_9[0][0… │
│                     │ (None, 256),      │            │ lstm[0][1],       │
│                     │ (None, 256)]      │            │ lstm[0][2]        │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_3 (Dense)     │ (None, 49, 3000)  │    771,000 │ lstm_1[0][0]      │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 21,447,736 (81.82 MB)

 Trainable params: 6,802,360 (25.95 MB)

 Non-trainable params: 14,645,376 (55.87 MB)

Epoch 1/8
47/47 ━━━━━━━━━━━━━━━━━━━━ 9s 67ms/step - accuracy: 0.0539 - loss: 6.9268 - val_accuracy: 0.0550 - val_loss: 6.3812
Epoch 2/8
47/47 ━━━━━━━━━━━━━━━━━━━━ 2s 49ms/step - accuracy: 0.0744 - loss: 6.3123 - val_accuracy: 0.0739 - val_loss: 6.2882
Epoch 3/8
47/47 ━━━━━━━━━━━━━━━━━━━━ 2s 48ms/step - accuracy: 0.0848 - loss: 6.2230 - val_accuracy: 0.0829 - val_loss: 6.2405
Epoch 4/8
47/47 ━━━━━━━━━━━━━━━━━━━━ 2s 48ms/step - accuracy: 0.0904 - loss: 6.1651 - val_accuracy: 0.0954 - val_loss: 6.2013
Epoch 5/8
47/47 ━━━━━━━━━━━━━━━━━━━━ 2s 48ms/step - accuracy: 0.1004 - loss: 6.1144 - val_accuracy: 0.0970 - val_loss: 6.1669
Epoch 6/8
47/47 ━━━━━━━━━━━━━━━━━━━━ 2s 51ms/step - accuracy: 0.1141 - loss: 6.0599 - val_accuracy: 0.1137 - val_loss: 6.1258
Epoch 7/8
47/47 ━━━━━━━━━━━━━━━━━━━━ 2s 51ms/step - accuracy: 0.1237 - loss: 6.0008 - val_accuracy: 0.1201 - val_loss: 6.0784
Epoch 8/8
47/47 ━━━━━━━━━━━━━━━━━━━━ 2s 48ms/step - accuracy: 0.1270 - loss: 5.9395 - val_accuracy: 0.1261 - val_loss:

In [ ]:
#Inference and Evaluate
enc_inp_oh_lstm = model_oh_lstm.input[0]
enc_emb_oh_lstm = model_oh_lstm.layers[2].output

encoder_model_oh_lstm, decoder_model_oh_lstm = build_inference_models(
    cell_type='lstm',
    encoder_inputs=enc_inp_oh_lstm,
    enc_emb=enc_emb_oh_lstm,
    dec_emb_layer=dec_emb_layer_oh_lstm,
    decoder_dense=dec_dense_oh_lstm,
    hidden_dim=HIDDEN_DIM,
    max_dec_len=max_decoder_len,
    encoder_states_tensors=model_oh_lstm.get_layer(index=4).output
)

print("\nSample: One-Hot + LSTM")
for i in range(1):
    print("\n------")
    print("INPUT:", X_test.iloc[i])
    seq = X_test_pad_oh[i].reshape(1, max_encoder_len)
    print("PREDICTED:", decode_sequence(seq, encoder_model_oh_lstm,
                                        decoder_model_oh_lstm, 'lstm', max_decoder_len))

all_scores['OneHot_LSTM'] = evaluate_bleu(
    encoder_model_oh_lstm, decoder_model_oh_lstm,
    'lstm', 'One-Hot + LSTM',
    X_test_pad_oh, oh_target_tokenizer
)


Sample: One-Hot + LSTM

------
INPUT: ['عرف', 'ادب']
PREDICTED: 'قرب' 'شكل' 'شمل' 'او' 'عمد' 'زمن' 'عزز' 'مثل' 'خلل' 'ناخ' 'شخص' 'جمع' 'حدث' start 'ورد' 'الي' 'في' 'لنه' 'بشر' 'نبت' 'قدم' 'مصر'

BLEU Score [One-Hot + LSTM]: 0.0003


In [ ]:
print("One-Hot + GRU")

model_oh_gru, dec_emb_layer_oh_gru, dec_dense_oh_gru = build_seq2seq_external_emb(
    cell_type='gru',
    encoder_emb_matrix=encoder_onehot_matrix,
    decoder_emb_matrix=decoder_onehot_matrix,
    max_enc_len=max_encoder_len,
    max_dec_len=max_decoder_len,
    embedding_dim=oh_encoder_vocab_size,
    hidden_dim=HIDDEN_DIM,
    decoder_vocab_size=oh_decoder_vocab_size
)
model_oh_gru.summary()

model_oh_gru.fit(
    [X_train_pad_oh, decoder_input_data_oh],
    decoder_target_data_oh,
    batch_size=BATCH_SIZE,
    epochs=EPOCHS,
    validation_split=0.2
)

One-Hot + GRU


Model: "functional_14"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_21      │ (None, 150)       │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ input_layer_22      │ (None, 49)        │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_10        │ (None, 150, 2376) │  5,645,376 │ input_layer_21[0… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ not_equal_14        │ (None, 150)       │          0 │ input_layer_21[0… │
│ (NotEqual)          │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_11        │ (None, 49, 3000)  │  9,000,000 │ input_layer_22[0… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ gru (GRU)           │ [(None, 256),     │  2,022,912 │ embedding_10[0][… │
│                     │ (None, 256)]      │            │ not_equal_14[0][… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ gru_1 (GRU)         │ [(None, 49, 256), │  2,502,144 │ embedding_11[0][… │
│                     │ (None, 256)]      │            │ gru[0][1]         │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_4 (Dense)     │ (None, 49, 3000)  │    771,000 │ gru_1[0][0]       │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 19,941,432 (76.07 MB)

 Trainable params: 5,296,056 (20.20 MB)

 Non-trainable params: 14,645,376 (55.87 MB)

Epoch 1/8
47/47 ━━━━━━━━━━━━━━━━━━━━ 5s 58ms/step - accuracy: 0.0589 - loss: 6.9932 - val_accuracy: 0.0521 - val_loss: 6.4255
Epoch 2/8
47/47 ━━━━━━━━━━━━━━━━━━━━ 2s 44ms/step - accuracy: 0.0775 - loss: 6.3354 - val_accuracy: 0.0767 - val_loss: 6.3116
Epoch 3/8
47/47 ━━━━━━━━━━━━━━━━━━━━ 2s 44ms/step - accuracy: 0.0908 - loss: 6.2531 - val_accuracy: 0.0872 - val_loss: 6.2727
Epoch 4/8
47/47 ━━━━━━━━━━━━━━━━━━━━ 2s 44ms/step - accuracy: 0.1037 - loss: 6.1894 - val_accuracy: 0.1084 - val_loss: 6.2243
Epoch 5/8
47/47 ━━━━━━━━━━━━━━━━━━━━ 2s 44ms/step - accuracy: 0.1182 - loss: 6.1125 - val_accuracy: 0.1200 - val_loss: 6.1427
Epoch 6/8
47/47 ━━━━━━━━━━━━━━━━━━━━ 3s 54ms/step - accuracy: 0.1288 - loss: 5.9996 - val_accuracy: 0.1299 - val_loss: 6.0453
Epoch 7/8
47/47 ━━━━━━━━━━━━━━━━━━━━ 2s 46ms/step - accuracy: 0.1386 - loss: 5.8606 - val_accuracy: 0.1383 - val_loss: 5.9249
Epoch 8/8
47/47 ━━━━━━━━━━━━━━━━━━━━ 2s 44ms/step - accuracy: 0.1516 - loss: 5.6964 - val_accuracy: 0.1529 - val_loss:

In [ ]:
#Inference and Evaluate
enc_inp_oh_gru = model_oh_gru.input[0]
enc_emb_oh_gru = model_oh_gru.layers[2].output

encoder_model_oh_gru, decoder_model_oh_gru = build_inference_models(
    cell_type='gru',
    encoder_inputs=enc_inp_oh_gru,
    enc_emb=enc_emb_oh_gru,
    dec_emb_layer=dec_emb_layer_oh_gru,
    decoder_dense=dec_dense_oh_gru,
    hidden_dim=HIDDEN_DIM,
    max_dec_len=max_decoder_len,
    encoder_states_tensors=model_oh_gru.get_layer(index=4).output
)

print("\nSample: One-Hot + GRU")
for i in range(3):
    print("\n----------")
    print("INPUT:", X_test.iloc[i])
    seq = X_test_pad_oh[i].reshape(1, max_encoder_len)
    print("PREDICTED:", decode_sequence(seq, encoder_model_oh_gru,
                                        decoder_model_oh_gru, 'gru', max_decoder_len))

all_scores['OneHot_GRU'] = evaluate_bleu(
    encoder_model_oh_gru, decoder_model_oh_gru,
    'gru', 'One-Hot + GRU',
    X_test_pad_oh, oh_target_tokenizer
)


Sample: One-Hot + GRU

----------
INPUT: ['عرف', 'ادب']
PREDICTED: 'شمل' 'تنع' 'عزز' 'من' 'الي' 'علم' 'عبر' 'جعل' 'ركز' 'نفس' 'طقه' 'رفع' 'فاد' 'هي' 'عمد' 'زمن' 'ساح' 'نبت' 'عمل' 'قلل' 'في' 'قرب' 'فرد'

----------
INPUT: ['عم', 'فكر', 'عند', 'قرء']
PREDICTED: 'عصر' 'تنع' 'قدم' 'اهم' 'علم' 'الي' 'طقه' 'مرض' 'عمد' 'او' 'في' 'من' 'دول' 'كفي' 'وهي' 'شمل' 'طبب' 'تطر' start 'زمن' 'سعد'

----------
INPUT: ['لمذ', 'ضرب', 'ضرب', 'رقم', 'صفر', 'يسو', 'وحد']
PREDICTED: 'شكل' 'اثر' 'حرف' 'من' 'سعد' 'الي' 'طقه' 'مرض' 'الذي' 'مصر' 'بين' 'ان' 'عن' 'شمل' 'عمد' 'فاد' 'قدر' 'نيل' 'نتج' 'او' 'زيد' 'فرد' 'فيه' 'قدم' start 'زمن' 'قرب' 'عرف' 'علي'

BLEU Score [One-Hot + GRU]: 0.0003


# **FastText**

In [ ]:
train_tokens = [text.split() for text in X_train]

ft_model = FastText(train_tokens, vector_size=EMBEDDING_DIM,
                    window=5, sg=1, min_count=1, epochs=10)

def build_embedding_matrix(word_index, ft_model, vocab_size, embedding_dim):
    matrix = np.zeros((vocab_size, embedding_dim), dtype='float32')
    for word, idx in word_index.items():
        matrix[idx] = ft_model.wv[word]
    return matrix

encoder_ft_matrix = build_embedding_matrix(
    input_tokenizer.word_index, ft_model, encoder_vocab_size, EMBEDDING_DIM)
decoder_ft_matrix = build_embedding_matrix(
    target_tokenizer.word_index, ft_model, decoder_vocab_size, EMBEDDING_DIM)

print("FastText encoder matrix shape:", encoder_ft_matrix.shape)
print("FastText decoder matrix shape:", decoder_ft_matrix.shape)

FastText encoder matrix shape: (2375, 128)
FastText decoder matrix shape: (4083, 128)


In [ ]:
print("FastText + RNN")

model_ft_rnn, dec_emb_layer_ft_rnn, dec_dense_ft_rnn = build_seq2seq_external_emb(
    cell_type='rnn',
    encoder_emb_matrix=encoder_ft_matrix,
    decoder_emb_matrix=decoder_ft_matrix,
    max_enc_len=max_encoder_len,
    max_dec_len=max_decoder_len,
    embedding_dim=EMBEDDING_DIM,
    hidden_dim=HIDDEN_DIM,
    decoder_vocab_size=decoder_vocab_size
)
model_ft_rnn.summary()

model_ft_rnn.fit(
    [X_train_pad, decoder_input_data],
    decoder_target_data,
    batch_size=BATCH_SIZE,
    epochs=EPOCHS,
    validation_split=0.2
)

FastText + RNN


Model: "functional_17"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_25      │ (None, 150)       │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ input_layer_26      │ (None, 49)        │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_12        │ (None, 150, 128)  │    304,000 │ input_layer_25[0… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ not_equal_17        │ (None, 150)       │          0 │ input_layer_25[0… │
│ (NotEqual)          │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_13        │ (None, 49, 128)   │    522,624 │ input_layer_26[0… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ simple_rnn_11       │ [(None, 256),     │     98,560 │ embedding_12[0][… │
│ (SimpleRNN)         │ (None, 256)]      │            │ not_equal_17[0][… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ simple_rnn_12       │ [(None, 49, 256), │     98,560 │ embedding_13[0][… │
│ (SimpleRNN)         │ (None, 256)]      │            │ simple_rnn_11[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_5 (Dense)     │ (None, 49, 4083)  │  1,049,331 │ simple_rnn_12[0]… │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 2,073,075 (7.91 MB)

 Trainable params: 1,246,451 (4.75 MB)

 Non-trainable params: 826,624 (3.15 MB)

Epoch 1/8
47/47 ━━━━━━━━━━━━━━━━━━━━ 10s 132ms/step - accuracy: 0.0505 - loss: 6.8738 - val_accuracy: 0.0534 - val_loss: 6.5724
Epoch 2/8
47/47 ━━━━━━━━━━━━━━━━━━━━ 4s 31ms/step - accuracy: 0.0639 - loss: 6.4460 - val_accuracy: 0.0713 - val_loss: 6.5035
Epoch 3/8
47/47 ━━━━━━━━━━━━━━━━━━━━ 1s 29ms/step - accuracy: 0.0814 - loss: 6.3644 - val_accuracy: 0.0815 - val_loss: 6.4608
Epoch 4/8
47/47 ━━━━━━━━━━━━━━━━━━━━ 1s 29ms/step - accuracy: 0.0931 - loss: 6.3013 - val_accuracy: 0.1029 - val_loss: 6.4220
Epoch 5/8
47/47 ━━━━━━━━━━━━━━━━━━━━ 1s 29ms/step - accuracy: 0.1009 - loss: 6.2306 - val_accuracy: 0.1122 - val_loss: 6.3710
Epoch 6/8
47/47 ━━━━━━━━━━━━━━━━━━━━ 1s 30ms/step - accuracy: 0.1101 - loss: 6.1383 - val_accuracy: 0.1136 - val_loss: 6.2949
Epoch 7/8
47/47 ━━━━━━━━━━━━━━━━━━━━ 1s 31ms/step - accuracy: 0.1157 - loss: 6.0366 - val_accuracy: 0.1156 - val_loss: 6.2247
Epoch 8/8
47/47 ━━━━━━━━━━━━━━━━━━━━ 2s 29ms/step - accuracy: 0.1221 - loss: 5.9432 - val_accuracy: 0.1194 - val_los

In [ ]:
enc_inp_ft_rnn = model_ft_rnn.input[0]
enc_emb_ft_rnn = model_ft_rnn.layers[2].output

encoder_model_ft_rnn, decoder_model_ft_rnn = build_inference_models(
    cell_type='rnn',
    encoder_inputs=enc_inp_ft_rnn,
    enc_emb=enc_emb_ft_rnn,
    dec_emb_layer=dec_emb_layer_ft_rnn,
    decoder_dense=dec_dense_ft_rnn,
    hidden_dim=HIDDEN_DIM,
    max_dec_len=max_decoder_len,
    encoder_states_tensors=model_ft_rnn.get_layer(index=4).output
)

print("\nSample: FastText + RNN")
for i in range(3):
    print("\n-------------")
    print("INPUT:", X_test.iloc[i])
    seq = X_test_pad[i].reshape(1, max_encoder_len)
    print("PREDICTED:", decode_sequence(seq, encoder_model_ft_rnn,
                                        decoder_model_ft_rnn, 'rnn', max_decoder_len))

all_scores['FastText_RNN'] = evaluate_bleu(encoder_model_ft_rnn, decoder_model_ft_rnn,
                                            'rnn', 'FastText + RNN')


Sample: FastText + RNN

-------------
INPUT: ['عرف', 'ادب']
PREDICTED: 'عزز' 'نمه' 'يمك' 'سمه' 'جبه' 'علي' 'برت' 'تكم' 'ننم' '1891' 'صطلح' 'قهه' 'عثر' 'حوس' 'قصي' '350' 'تتك' 'ردف' 'خردل'

-------------
INPUT: ['عم', 'فكر', 'عند', 'قرء']
PREDICTED: 'فعل' 'ولي' 'لا' 'هو' 'شمل' 'جود' 'اتج' 'ماض' 'تعد' 'اجد' 'وضق' 'قليل' 'يشك' 'وجد' 'يتك' 'بني' 'تقع' 'سبب' 'حرر' 'مالديف' 'كورم' 'يمكن' '90' 'سجه' 'اسك' 'تكم' 'متص' 'شيبس' 'فنن' 'جبه' 'عار' 'فمن' '1986' 'غريسوم' 'جاي' '4458' 'فرس' 'دسم' 'حبب' 'تتك' 'بدن' 'جينوم' 'خفي'

-------------
INPUT: ['لمذ', 'ضرب', 'ضرب', 'رقم', 'صفر', 'يسو', 'وحد']
PREDICTED: 'خلل' 'مدن' 'هو' 'خلف' 'خرع' 'نسق' 'تم' 'وزن' 'نصم' 'عدن' 'فيبوناتشي' 'وسم' 'عذراء' 'سعي' 'لول' 'يصل' 'وجد' 'غنغ' 'ليس' 'سبب' 'ظهر' 'بحر' 'يعج' 'شمل' 'كنولوج' 'خجل'

BLEU Score [FastText + RNN]: 0.0000


In [ ]:
print("FastText + LSTM")

model_ft_lstm, dec_emb_layer_ft_lstm, dec_dense_ft_lstm = build_seq2seq_external_emb(
    cell_type='lstm',
    encoder_emb_matrix=encoder_ft_matrix,
    decoder_emb_matrix=decoder_ft_matrix,
    max_enc_len=max_encoder_len,
    max_dec_len=max_decoder_len,
    embedding_dim=EMBEDDING_DIM,
    hidden_dim=HIDDEN_DIM,
    decoder_vocab_size=decoder_vocab_size
)
model_ft_lstm.summary()

model_ft_lstm.fit(
    [X_train_pad, decoder_input_data],
    decoder_target_data,
    batch_size=BATCH_SIZE,
    epochs=EPOCHS,
    validation_split=0.2
)

FastText + LSTM


Model: "functional_20"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_29      │ (None, 150)       │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ input_layer_30      │ (None, 49)        │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_14        │ (None, 150, 128)  │    304,000 │ input_layer_29[0… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ not_equal_20        │ (None, 150)       │          0 │ input_layer_29[0… │
│ (NotEqual)          │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_15        │ (None, 49, 128)   │    522,624 │ input_layer_30[0… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm_3 (LSTM)       │ [(None, 256),     │    394,240 │ embedding_14[0][… │
│                     │ (None, 256),      │            │ not_equal_20[0][… │
│                     │ (None, 256)]      │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm_4 (LSTM)       │ [(None, 49, 256), │    394,240 │ embedding_15[0][… │
│                     │ (None, 256),      │            │ lstm_3[0][1],     │
│                     │ (None, 256)]      │            │ lstm_3[0][2]      │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_6 (Dense)     │ (None, 49, 4083)  │  1,049,331 │ lstm_4[0][0]      │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 2,664,435 (10.16 MB)

 Trainable params: 1,837,811 (7.01 MB)

 Non-trainable params: 826,624 (3.15 MB)

Epoch 1/8
47/47 ━━━━━━━━━━━━━━━━━━━━ 4s 51ms/step - accuracy: 0.0518 - loss: 6.9908 - val_accuracy: 0.0593 - val_loss: 6.5345
Epoch 2/8
47/47 ━━━━━━━━━━━━━━━━━━━━ 2s 39ms/step - accuracy: 0.0690 - loss: 6.4016 - val_accuracy: 0.0748 - val_loss: 6.4605
Epoch 3/8
47/47 ━━━━━━━━━━━━━━━━━━━━ 2s 47ms/step - accuracy: 0.0870 - loss: 6.3198 - val_accuracy: 0.0933 - val_loss: 6.4459
Epoch 4/8
47/47 ━━━━━━━━━━━━━━━━━━━━ 2s 41ms/step - accuracy: 0.0888 - loss: 6.2728 - val_accuracy: 0.0879 - val_loss: 6.4086
Epoch 5/8
47/47 ━━━━━━━━━━━━━━━━━━━━ 2s 39ms/step - accuracy: 0.0964 - loss: 6.2246 - val_accuracy: 0.1000 - val_loss: 6.3839
Epoch 6/8
47/47 ━━━━━━━━━━━━━━━━━━━━ 2s 39ms/step - accuracy: 0.1058 - loss: 6.1689 - val_accuracy: 0.1037 - val_loss: 6.3423
Epoch 7/8
47/47 ━━━━━━━━━━━━━━━━━━━━ 2s 39ms/step - accuracy: 0.1119 - loss: 6.1032 - val_accuracy: 0.1129 - val_loss: 6.2924
Epoch 8/8
47/47 ━━━━━━━━━━━━━━━━━━━━ 2s 39ms/step - accuracy: 0.1163 - loss: 6.0266 - val_accuracy: 0.1155 - val_loss:

In [ ]:
enc_inp_ft_lstm = model_ft_lstm.input[0]
enc_emb_ft_lstm = model_ft_lstm.layers[2].output

encoder_model_ft_lstm, decoder_model_ft_lstm = build_inference_models(
    cell_type='lstm',
    encoder_inputs=enc_inp_ft_lstm,
    enc_emb=enc_emb_ft_lstm,
    dec_emb_layer=dec_emb_layer_ft_lstm,
    decoder_dense=dec_dense_ft_lstm,
    hidden_dim=HIDDEN_DIM,
    max_dec_len=max_decoder_len,
    encoder_states_tensors=model_ft_lstm.get_layer(index=4).output
)

print("\nSample: FastText + LSTM")
for i in range(3):
    print("\n-----------")
    print("INPUT:", X_test.iloc[i])
    seq = X_test_pad[i].reshape(1, max_encoder_len)
    print("PREDICTED:", decode_sequence(seq, encoder_model_ft_lstm,
                                        decoder_model_ft_lstm, 'lstm', max_decoder_len))

all_scores['FastText_LSTM'] = evaluate_bleu(encoder_model_ft_lstm, decoder_model_ft_lstm,
                                             'lstm', 'FastText + LSTM')


Sample: FastText + LSTM

-----------
INPUT: ['عرف', 'ادب']
PREDICTED: 'عمد' 'اجد' 'احب' 'يقس' 'فيد' 'علم' 'خرع' 'تتك' 'فضل' 'سمد' 'متع' 'التي' 'خرج' 'بين' 'قرء' 'في'

-----------
INPUT: ['عم', 'فكر', 'عند', 'قرء']
PREDICTED: 'فيد' 'يمكن' 'حدث' 'عبر' 'تم' 'عمد' 'بدا' 'علم' 'في' 'سمد' 'تتك' 'ريس' 'خرع' 'متع' 'ان' 'الذي' 'التي' 'من' 'عمل' 'شكل' 'خرج' 'بين'

-----------
INPUT: ['لمذ', 'ضرب', 'ضرب', 'رقم', 'صفر', 'يسو', 'وحد']
PREDICTED: 'عدد' 'عبر' 'عمد' 'في' 'رفع' 'التي' 'ان' 'الي' 'بين' 'خرع' 'خرج' 'متع' 'من' 'سمد' 'تتك'

BLEU Score [FastText + LSTM]: 0.0000


In [ ]:
print("FastText + GRU")

model_ft_gru, dec_emb_layer_ft_gru, dec_dense_ft_gru = build_seq2seq_external_emb(
    cell_type='gru',
    encoder_emb_matrix=encoder_ft_matrix,
    decoder_emb_matrix=decoder_ft_matrix,
    max_enc_len=max_encoder_len,
    max_dec_len=max_decoder_len,
    embedding_dim=EMBEDDING_DIM,
    hidden_dim=HIDDEN_DIM,
    decoder_vocab_size=decoder_vocab_size
)
model_ft_gru.summary()

model_ft_gru.fit(
    [X_train_pad, decoder_input_data],
    decoder_target_data,
    batch_size=BATCH_SIZE,
    epochs=EPOCHS,
    validation_split=0.2
)

FastText + GRU


Model: "functional_23"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_34      │ (None, 150)       │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ input_layer_35      │ (None, 49)        │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_16        │ (None, 150, 128)  │    304,000 │ input_layer_34[0… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ not_equal_23        │ (None, 150)       │          0 │ input_layer_34[0… │
│ (NotEqual)          │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_17        │ (None, 49, 128)   │    522,624 │ input_layer_35[0… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ gru_3 (GRU)         │ [(None, 256),     │    296,448 │ embedding_16[0][… │
│                     │ (None, 256)]      │            │ not_equal_23[0][… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ gru_4 (GRU)         │ [(None, 49, 256), │    296,448 │ embedding_17[0][… │
│                     │ (None, 256)]      │            │ gru_3[0][1]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_7 (Dense)     │ (None, 49, 4083)  │  1,049,331 │ gru_4[0][0]       │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 2,468,851 (9.42 MB)

 Trainable params: 1,642,227 (6.26 MB)

 Non-trainable params: 826,624 (3.15 MB)

Epoch 1/8
47/47 ━━━━━━━━━━━━━━━━━━━━ 4s 49ms/step - accuracy: 0.0543 - loss: 7.0412 - val_accuracy: 0.0508 - val_loss: 6.5894
Epoch 2/8
47/47 ━━━━━━━━━━━━━━━━━━━━ 2s 44ms/step - accuracy: 0.0829 - loss: 6.4296 - val_accuracy: 0.0845 - val_loss: 6.5042
Epoch 3/8
47/47 ━━━━━━━━━━━━━━━━━━━━ 2s 39ms/step - accuracy: 0.0910 - loss: 6.3458 - val_accuracy: 0.0924 - val_loss: 6.4658
Epoch 4/8
47/47 ━━━━━━━━━━━━━━━━━━━━ 2s 39ms/step - accuracy: 0.0991 - loss: 6.2692 - val_accuracy: 0.0970 - val_loss: 6.3973
Epoch 5/8
47/47 ━━━━━━━━━━━━━━━━━━━━ 2s 38ms/step - accuracy: 0.1088 - loss: 6.1550 - val_accuracy: 0.1086 - val_loss: 6.3166
Epoch 6/8
47/47 ━━━━━━━━━━━━━━━━━━━━ 2s 38ms/step - accuracy: 0.1174 - loss: 6.0466 - val_accuracy: 0.1164 - val_loss: 6.2455
Epoch 7/8
47/47 ━━━━━━━━━━━━━━━━━━━━ 2s 38ms/step - accuracy: 0.1242 - loss: 5.9405 - val_accuracy: 0.1252 - val_loss: 6.1805
Epoch 8/8
47/47 ━━━━━━━━━━━━━━━━━━━━ 3s 49ms/step - accuracy: 0.1293 - loss: 5.8358 - val_accuracy: 0.1266 - val_loss:

In [ ]:
enc_inp_ft_gru = model_ft_gru.input[0]
enc_emb_ft_gru = model_ft_gru.layers[2].output

encoder_model_ft_gru, decoder_model_ft_gru = build_inference_models(
    cell_type='gru',
    encoder_inputs=enc_inp_ft_gru,
    enc_emb=enc_emb_ft_gru,
    dec_emb_layer=dec_emb_layer_ft_gru,
    decoder_dense=dec_dense_ft_gru,
    hidden_dim=HIDDEN_DIM,
    max_dec_len=max_decoder_len,
    encoder_states_tensors=model_ft_gru.get_layer(index=4).output
)

print("\nSample: FastText + GRU")
for i in range(3):
    print("\n---------------")
    print("INPUT:", X_test.iloc[i])
    seq = X_test_pad[i].reshape(1, max_encoder_len)
    print("PREDICTED:", decode_sequence(seq, encoder_model_ft_gru,
                                        decoder_model_ft_gru, 'gru', max_decoder_len))

all_scores['FastText_GRU'] = evaluate_bleu(encoder_model_ft_gru, decoder_model_ft_gru,
                                            'gru', 'FastText + GRU')


Sample: FastText + GRU

---------------
INPUT: ['عرف', 'ادب']
PREDICTED: 'اجد' 'احب' 'فيد' 'علم' 'رفع' 'شمل' 'فضل' 'خرع' 'هو' 'متع' 'سمد' 'وسس' 'تتك' 'عدد' 'كتب' 'حدث' 'صنع' 'كشف' 'خرج' 'من'

---------------
INPUT: ['عم', 'فكر', 'عند', 'قرء']
PREDICTED: 'فيد' 'تكم' 'نجح' 'اهم' 'عبر' 'في' 'مرض' 'خرج' 'علم' 'خرع' 'هو' 'سمد' 'كشف' 'تتك' 'احب' 'تحس' 'شمل' 'متع' 'حدث' 'رفع' 'واع'

---------------
INPUT: ['لمذ', 'ضرب', 'ضرب', 'رقم', 'صفر', 'يسو', 'وحد']
PREDICTED: 'عبر' 'بدا' 'ان' 'يمكن' 'اجد' 'تتك' 'خرج' 'وسس' 'عدد' 'وسع' 'ولي' 'هار' 'هو' 'سمد' 'علم' 'خرع' 'شمل'

BLEU Score [FastText + GRU]: 0.0000


# **E5 Embeddings**

In [ ]:
e5_tokenizer  = AutoTokenizer.from_pretrained("intfloat/multilingual-e5-base")
e5_model      = AutoModel.from_pretrained("intfloat/multilingual-e5-base")
e5_model.eval()

e5_embed_table = e5_model.embeddings.word_embeddings.weight.detach().cpu().numpy()
E5_DIM = e5_embed_table.shape[1]   # 768
print("E5 embedding table shape:", e5_embed_table.shape)

def build_e5_matrix(word_index, e5_tokenizer, e5_embed_table, vocab_size):
    matrix = np.zeros((vocab_size, e5_embed_table.shape[1]), dtype='float32')
    for word, idx in word_index.items():
        e5_ids = e5_tokenizer.encode(word, add_special_tokens=False)
        if e5_ids:
            matrix[idx] = np.mean(e5_embed_table[e5_ids], axis=0)
    return matrix

encoder_e5_matrix = build_e5_matrix(
    input_tokenizer.word_index, e5_tokenizer, e5_embed_table, encoder_vocab_size)
decoder_e5_matrix = build_e5_matrix(
    target_tokenizer.word_index, e5_tokenizer, e5_embed_table, decoder_vocab_size)

print("E5 encoder matrix shape:", encoder_e5_matrix.shape)
print("E5 decoder matrix shape:", decoder_e5_matrix.shape)

# Free GPU memory
del e5_model
import torch, gc
torch.cuda.empty_cache()
gc.collect()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/694 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/418 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/280 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.11G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

E5 embedding table shape: (250002, 768)
E5 encoder matrix shape: (2375, 768)
E5 decoder matrix shape: (4083, 768)


1753

In [ ]:
print("E5 + RNN")

model_e5_rnn, dec_emb_layer_e5_rnn, dec_dense_e5_rnn = build_seq2seq_external_emb(
    cell_type='rnn',
    encoder_emb_matrix=encoder_e5_matrix,
    decoder_emb_matrix=decoder_e5_matrix,
    max_enc_len=max_encoder_len,
    max_dec_len=max_decoder_len,
    embedding_dim=E5_DIM,
    hidden_dim=HIDDEN_DIM,
    decoder_vocab_size=decoder_vocab_size
)
model_e5_rnn.summary()

model_e5_rnn.fit(
    [X_train_pad, decoder_input_data],
    decoder_target_data,
    batch_size=BATCH_SIZE,
    epochs=EPOCHS,
    validation_split=0.2
)

#Inference & BLEU

enc_inp_e5_rnn = model_e5_rnn.input[0]
enc_emb_e5_rnn = model_e5_rnn.layers[2].output

encoder_model_e5_rnn, decoder_model_e5_rnn = build_inference_models(
    cell_type='rnn',
    encoder_inputs=enc_inp_e5_rnn,
    enc_emb=enc_emb_e5_rnn,
    dec_emb_layer=dec_emb_layer_e5_rnn,
    decoder_dense=dec_dense_e5_rnn,
    hidden_dim=HIDDEN_DIM,
    max_dec_len=max_decoder_len,
    encoder_states_tensors=model_e5_rnn.get_layer(index=4).output
)

print("\nSample: E5 + RNN")
for i in range(3):
    print("\n---------------")
    print("INPUT:", X_test.iloc[i])
    seq = X_test_pad[i].reshape(1, max_encoder_len)
    print("PREDICTED:", decode_sequence(seq, encoder_model_e5_rnn,
                                        decoder_model_e5_rnn, 'rnn', max_decoder_len))

all_scores['E5_RNN'] = evaluate_bleu(encoder_model_e5_rnn, decoder_model_e5_rnn,
                                      'rnn', 'E5 + RNN')

E5 + RNN


Model: "functional_26"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_38      │ (None, 150)       │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ input_layer_39      │ (None, 49)        │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_18        │ (None, 150, 768)  │  1,824,000 │ input_layer_38[0… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ not_equal_26        │ (None, 150)       │          0 │ input_layer_38[0… │
│ (NotEqual)          │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_19        │ (None, 49, 768)   │  3,135,744 │ input_layer_39[0… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ simple_rnn_14       │ [(None, 256),     │    262,400 │ embedding_18[0][… │
│ (SimpleRNN)         │ (None, 256)]      │            │ not_equal_26[0][… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ simple_rnn_15       │ [(None, 49, 256), │    262,400 │ embedding_19[0][… │
│ (SimpleRNN)         │ (None, 256)]      │            │ simple_rnn_14[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_8 (Dense)     │ (None, 49, 4083)  │  1,049,331 │ simple_rnn_15[0]… │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 6,533,875 (24.92 MB)

 Trainable params: 1,574,131 (6.00 MB)

 Non-trainable params: 4,959,744 (18.92 MB)

Epoch 1/8
47/47 ━━━━━━━━━━━━━━━━━━━━ 11s 125ms/step - accuracy: 0.0960 - loss: 6.6937 - val_accuracy: 0.1043 - val_loss: 6.3461
Epoch 2/8
47/47 ━━━━━━━━━━━━━━━━━━━━ 2s 38ms/step - accuracy: 0.1142 - loss: 6.2240 - val_accuracy: 0.1156 - val_loss: 6.2984
Epoch 3/8
47/47 ━━━━━━━━━━━━━━━━━━━━ 2s 40ms/step - accuracy: 0.1212 - loss: 6.1483 - val_accuracy: 0.1190 - val_loss: 6.2706
Epoch 4/8
47/47 ━━━━━━━━━━━━━━━━━━━━ 2s 42ms/step - accuracy: 0.1213 - loss: 6.0961 - val_accuracy: 0.1164 - val_loss: 6.2495
Epoch 5/8
47/47 ━━━━━━━━━━━━━━━━━━━━ 2s 38ms/step - accuracy: 0.1243 - loss: 6.0392 - val_accuracy: 0.1249 - val_loss: 6.2311
Epoch 6/8
47/47 ━━━━━━━━━━━━━━━━━━━━ 2s 38ms/step - accuracy: 0.1272 - loss: 5.9859 - val_accuracy: 0.1290 - val_loss: 6.1785
Epoch 7/8
47/47 ━━━━━━━━━━━━━━━━━━━━ 2s 38ms/step - accuracy: 0.1344 - loss: 5.9118 - val_accuracy: 0.1329 - val_loss: 6.1379
Epoch 8/8
47/47 ━━━━━━━━━━━━━━━━━━━━ 2s 38ms/step - accuracy: 0.1378 - loss: 5.8336 - val_accuracy: 0.1369 - val_los

In [ ]:
print("E5 + LSTM")

model_e5_lstm, dec_emb_layer_e5_lstm, dec_dense_e5_lstm = build_seq2seq_external_emb(
    cell_type='lstm',
    encoder_emb_matrix=encoder_e5_matrix,
    decoder_emb_matrix=decoder_e5_matrix,
    max_enc_len=max_encoder_len,
    max_dec_len=max_decoder_len,
    embedding_dim=E5_DIM,
    hidden_dim=HIDDEN_DIM,
    decoder_vocab_size=decoder_vocab_size
)
model_e5_lstm.summary()

model_e5_lstm.fit(
    [X_train_pad, decoder_input_data],
    decoder_target_data,
    batch_size=BATCH_SIZE,
    epochs=EPOCHS,
    validation_split=0.2
)

#Inference & BLEU

enc_inp_e5_lstm = model_e5_lstm.input[0]
enc_emb_e5_lstm = model_e5_lstm.layers[2].output

encoder_model_e5_lstm, decoder_model_e5_lstm = build_inference_models(
    cell_type='lstm',
    encoder_inputs=enc_inp_e5_lstm,
    enc_emb=enc_emb_e5_lstm,
    dec_emb_layer=dec_emb_layer_e5_lstm,
    decoder_dense=dec_dense_e5_lstm,
    hidden_dim=HIDDEN_DIM,
    max_dec_len=max_decoder_len,
    encoder_states_tensors=model_e5_lstm.get_layer(index=4).output
)

print("\nSample: E5 + LSTM")
for i in range(3):
    print("\n----------------")
    print("INPUT:", X_test.iloc[i])
    seq = X_test_pad[i].reshape(1, max_encoder_len)
    print("PREDICTED:", decode_sequence(seq, encoder_model_e5_lstm,
                                        decoder_model_e5_lstm, 'lstm', max_decoder_len))

all_scores['E5_LSTM'] = evaluate_bleu(encoder_model_e5_lstm, decoder_model_e5_lstm,
                                       'lstm', 'E5 + LSTM')

E5 + LSTM


Model: "functional_29"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_42      │ (None, 150)       │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ input_layer_43      │ (None, 49)        │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_20        │ (None, 150, 768)  │  1,824,000 │ input_layer_42[0… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ not_equal_29        │ (None, 150)       │          0 │ input_layer_42[0… │
│ (NotEqual)          │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_21        │ (None, 49, 768)   │  3,135,744 │ input_layer_43[0… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm_6 (LSTM)       │ [(None, 256),     │  1,049,600 │ embedding_20[0][… │
│                     │ (None, 256),      │            │ not_equal_29[0][… │
│                     │ (None, 256)]      │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm_7 (LSTM)       │ [(None, 49, 256), │  1,049,600 │ embedding_21[0][… │
│                     │ (None, 256),      │            │ lstm_6[0][1],     │
│                     │ (None, 256)]      │            │ lstm_6[0][2]      │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_9 (Dense)     │ (None, 49, 4083)  │  1,049,331 │ lstm_7[0][0]      │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 8,108,275 (30.93 MB)

 Trainable params: 3,148,531 (12.01 MB)

 Non-trainable params: 4,959,744 (18.92 MB)

Epoch 1/8
47/47 ━━━━━━━━━━━━━━━━━━━━ 4s 53ms/step - accuracy: 0.0925 - loss: 6.7690 - val_accuracy: 0.1136 - val_loss: 6.4407
Epoch 2/8
47/47 ━━━━━━━━━━━━━━━━━━━━ 2s 43ms/step - accuracy: 0.1190 - loss: 6.2483 - val_accuracy: 0.1173 - val_loss: 6.2912
Epoch 3/8
47/47 ━━━━━━━━━━━━━━━━━━━━ 3s 53ms/step - accuracy: 0.1209 - loss: 6.1397 - val_accuracy: 0.1172 - val_loss: 6.2592
Epoch 4/8
47/47 ━━━━━━━━━━━━━━━━━━━━ 2s 45ms/step - accuracy: 0.1211 - loss: 6.0900 - val_accuracy: 0.1181 - val_loss: 6.2396
Epoch 5/8
47/47 ━━━━━━━━━━━━━━━━━━━━ 2s 43ms/step - accuracy: 0.1215 - loss: 6.0533 - val_accuracy: 0.1178 - val_loss: 6.2245
Epoch 6/8
47/47 ━━━━━━━━━━━━━━━━━━━━ 2s 43ms/step - accuracy: 0.1233 - loss: 6.0184 - val_accuracy: 0.1205 - val_loss: 6.2102
Epoch 7/8
47/47 ━━━━━━━━━━━━━━━━━━━━ 2s 43ms/step - accuracy: 0.1266 - loss: 5.9811 - val_accuracy: 0.1265 - val_loss: 6.1854
Epoch 8/8
47/47 ━━━━━━━━━━━━━━━━━━━━ 2s 43ms/step - accuracy: 0.1304 - loss: 5.9389 - val_accuracy: 0.1282 - val_loss:

In [ ]:
print("E5 + GRU")

model_e5_gru, dec_emb_layer_e5_gru, dec_dense_e5_gru = build_seq2seq_external_emb(
    cell_type='gru',
    encoder_emb_matrix=encoder_e5_matrix,
    decoder_emb_matrix=decoder_e5_matrix,
    max_enc_len=max_encoder_len,
    max_dec_len=max_decoder_len,
    embedding_dim=E5_DIM,
    hidden_dim=HIDDEN_DIM,
    decoder_vocab_size=decoder_vocab_size
)
model_e5_gru.summary()

model_e5_gru.fit(
    [X_train_pad, decoder_input_data],
    decoder_target_data,
    batch_size=BATCH_SIZE,
    epochs=EPOCHS,
    validation_split=0.2
)

#Inference & BLEU

enc_inp_e5_gru = model_e5_gru.input[0]
enc_emb_e5_gru = model_e5_gru.layers[2].output

encoder_model_e5_gru, decoder_model_e5_gru = build_inference_models(
    cell_type='gru',
    encoder_inputs=enc_inp_e5_gru,
    enc_emb=enc_emb_e5_gru,
    dec_emb_layer=dec_emb_layer_e5_gru,
    decoder_dense=dec_dense_e5_gru,
    hidden_dim=HIDDEN_DIM,
    max_dec_len=max_decoder_len,
    encoder_states_tensors=model_e5_gru.get_layer(index=4).output
)

print("\nSample: E5 + GRU")
for i in range(3):
    print("\n-------------")
    print("INPUT:", X_test.iloc[i])
    seq = X_test_pad[i].reshape(1, max_encoder_len)
    print("PREDICTED:", decode_sequence(seq, encoder_model_e5_gru,
                                        decoder_model_e5_gru, 'gru', max_decoder_len))

all_scores['E5_GRU'] = evaluate_bleu(encoder_model_e5_gru, decoder_model_e5_gru,
                                      'gru', 'E5 + GRU')

E5 + GRU


Model: "functional_32"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_47      │ (None, 150)       │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ input_layer_48      │ (None, 49)        │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_22        │ (None, 150, 768)  │  1,824,000 │ input_layer_47[0… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ not_equal_32        │ (None, 150)       │          0 │ input_layer_47[0… │
│ (NotEqual)          │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_23        │ (None, 49, 768)   │  3,135,744 │ input_layer_48[0… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ gru_6 (GRU)         │ [(None, 256),     │    787,968 │ embedding_22[0][… │
│                     │ (None, 256)]      │            │ not_equal_32[0][… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ gru_7 (GRU)         │ [(None, 49, 256), │    787,968 │ embedding_23[0][… │
│                     │ (None, 256)]      │            │ gru_6[0][1]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_10 (Dense)    │ (None, 49, 4083)  │  1,049,331 │ gru_7[0][0]       │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 7,585,011 (28.93 MB)

 Trainable params: 2,625,267 (10.01 MB)

 Non-trainable params: 4,959,744 (18.92 MB)

Epoch 1/8
47/47 ━━━━━━━━━━━━━━━━━━━━ 4s 53ms/step - accuracy: 0.0991 - loss: 6.7255 - val_accuracy: 0.1145 - val_loss: 6.4008
Epoch 2/8
47/47 ━━━━━━━━━━━━━━━━━━━━ 2s 41ms/step - accuracy: 0.1207 - loss: 6.2384 - val_accuracy: 0.1182 - val_loss: 6.3088
Epoch 3/8
47/47 ━━━━━━━━━━━━━━━━━━━━ 2s 45ms/step - accuracy: 0.1210 - loss: 6.1608 - val_accuracy: 0.1180 - val_loss: 6.2959
Epoch 4/8
47/47 ━━━━━━━━━━━━━━━━━━━━ 2s 45ms/step - accuracy: 0.1216 - loss: 6.1193 - val_accuracy: 0.1189 - val_loss: 6.2641
Epoch 5/8
47/47 ━━━━━━━━━━━━━━━━━━━━ 2s 42ms/step - accuracy: 0.1242 - loss: 6.0573 - val_accuracy: 0.1226 - val_loss: 6.2154
Epoch 6/8
47/47 ━━━━━━━━━━━━━━━━━━━━ 2s 41ms/step - accuracy: 0.1307 - loss: 5.9646 - val_accuracy: 0.1317 - val_loss: 6.1416
Epoch 7/8
47/47 ━━━━━━━━━━━━━━━━━━━━ 2s 42ms/step - accuracy: 0.1398 - loss: 5.8469 - val_accuracy: 0.1386 - val_loss: 6.0569
Epoch 8/8
47/47 ━━━━━━━━━━━━━━━━━━━━ 2s 42ms/step - accuracy: 0.1473 - loss: 5.7238 - val_accuracy: 0.1471 - val_loss:

# **AraBERT**

In [ ]:
arabert_tokenizer = AutoTokenizer.from_pretrained("aubmindlab/bert-base-arabertv02")
arabert_model     = AutoModel.from_pretrained("aubmindlab/bert-base-arabertv02")
arabert_model.eval()

arabert_embed_table = arabert_model.embeddings.word_embeddings.weight.detach().cpu().numpy()
ARABERT_DIM = arabert_embed_table.shape[1]  # 768
print("AraBERT embedding table shape:", arabert_embed_table.shape)

def build_arabert_matrix(word_index, tokenizer, embed_table, vocab_size):
    matrix = np.zeros((vocab_size, embed_table.shape[1]), dtype='float32')
    for word, idx in word_index.items():
        ids = tokenizer.encode(word, add_special_tokens=False)
        if ids:
            matrix[idx] = np.mean(embed_table[ids], axis=0)
    return matrix

encoder_arabert_matrix = build_arabert_matrix(
    input_tokenizer.word_index, arabert_tokenizer, arabert_embed_table, encoder_vocab_size)
decoder_arabert_matrix = build_arabert_matrix(
    target_tokenizer.word_index, arabert_tokenizer, arabert_embed_table, decoder_vocab_size)

print("AraBERT encoder matrix shape:", encoder_arabert_matrix.shape)
print("AraBERT decoder matrix shape:", decoder_arabert_matrix.shape)

del arabert_model
torch.cuda.empty_cache()
gc.collect()

config.json:   0%|          | 0.00/384 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/381 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/825k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.64M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/543M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: aubmindlab/bert-base-arabertv02
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


AraBERT embedding table shape: (64000, 768)
AraBERT encoder matrix shape: (2375, 768)
AraBERT decoder matrix shape: (4083, 768)


23042

In [ ]:
print("AraBERT + RNN")

model_ab_rnn, dec_emb_layer_ab_rnn, dec_dense_ab_rnn = build_seq2seq_external_emb(
    cell_type='rnn',
    encoder_emb_matrix=encoder_arabert_matrix,
    decoder_emb_matrix=decoder_arabert_matrix,
    max_enc_len=max_encoder_len,
    max_dec_len=max_decoder_len,
    embedding_dim=ARABERT_DIM,
    hidden_dim=HIDDEN_DIM,
    decoder_vocab_size=decoder_vocab_size
)
model_ab_rnn.summary()

model_ab_rnn.fit(
    [X_train_pad, decoder_input_data],
    decoder_target_data,
    batch_size=BATCH_SIZE,
    epochs=EPOCHS,
    validation_split=0.2
)

#Inference & BLEU

enc_inp_ab_rnn = model_ab_rnn.input[0]
enc_emb_ab_rnn = model_ab_rnn.layers[2].output

encoder_model_ab_rnn, decoder_model_ab_rnn = build_inference_models(
    cell_type='rnn',
    encoder_inputs=enc_inp_ab_rnn,
    enc_emb=enc_emb_ab_rnn,
    dec_emb_layer=dec_emb_layer_ab_rnn,
    decoder_dense=dec_dense_ab_rnn,
    hidden_dim=HIDDEN_DIM,
    max_dec_len=max_decoder_len,
    encoder_states_tensors=model_ab_rnn.get_layer(index=4).output
)

print("\nSample: AraBERT + RNN")
for i in range(3):
    print("\n------------")
    print("INPUT:", X_test.iloc[i])
    seq = X_test_pad[i].reshape(1, max_encoder_len)
    print("PREDICTED:", decode_sequence(seq, encoder_model_ab_rnn,
                                        decoder_model_ab_rnn, 'rnn', max_decoder_len))

all_scores['AraBERT_RNN'] = evaluate_bleu(encoder_model_ab_rnn, decoder_model_ab_rnn,
                                           'rnn', 'AraBERT + RNN')

AraBERT + RNN


Model: "functional_35"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_51      │ (None, 150)       │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ input_layer_52      │ (None, 49)        │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_24        │ (None, 150, 768)  │  1,824,000 │ input_layer_51[0… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ not_equal_35        │ (None, 150)       │          0 │ input_layer_51[0… │
│ (NotEqual)          │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_25        │ (None, 49, 768)   │  3,135,744 │ input_layer_52[0… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ simple_rnn_17       │ [(None, 256),     │    262,400 │ embedding_24[0][… │
│ (SimpleRNN)         │ (None, 256)]      │            │ not_equal_35[0][… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ simple_rnn_18       │ [(None, 49, 256), │    262,400 │ embedding_25[0][… │
│ (SimpleRNN)         │ (None, 256)]      │            │ simple_rnn_17[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_11 (Dense)    │ (None, 49, 4083)  │  1,049,331 │ simple_rnn_18[0]… │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 6,533,875 (24.92 MB)

 Trainable params: 1,574,131 (6.00 MB)

 Non-trainable params: 4,959,744 (18.92 MB)

Epoch 1/8
47/47 ━━━━━━━━━━━━━━━━━━━━ 10s 137ms/step - accuracy: 0.0599 - loss: 6.8121 - val_accuracy: 0.0534 - val_loss: 6.4998
Epoch 2/8
47/47 ━━━━━━━━━━━━━━━━━━━━ 2s 40ms/step - accuracy: 0.1101 - loss: 6.3084 - val_accuracy: 0.1141 - val_loss: 6.3475
Epoch 3/8
47/47 ━━━━━━━━━━━━━━━━━━━━ 2s 40ms/step - accuracy: 0.1203 - loss: 6.1796 - val_accuracy: 0.1168 - val_loss: 6.2918
Epoch 4/8
47/47 ━━━━━━━━━━━━━━━━━━━━ 2s 40ms/step - accuracy: 0.1214 - loss: 6.1190 - val_accuracy: 0.1182 - val_loss: 6.2697
Epoch 5/8
47/47 ━━━━━━━━━━━━━━━━━━━━ 2s 42ms/step - accuracy: 0.1212 - loss: 6.0771 - val_accuracy: 0.1192 - val_loss: 6.2595
Epoch 6/8
47/47 ━━━━━━━━━━━━━━━━━━━━ 2s 42ms/step - accuracy: 0.1217 - loss: 6.0406 - val_accuracy: 0.1169 - val_loss: 6.2522
Epoch 7/8
47/47 ━━━━━━━━━━━━━━━━━━━━ 2s 40ms/step - accuracy: 0.1219 - loss: 6.0038 - val_accuracy: 0.1198 - val_loss: 6.2418
Epoch 8/8
47/47 ━━━━━━━━━━━━━━━━━━━━ 2s 40ms/step - accuracy: 0.1244 - loss: 5.9630 - val_accuracy: 0.1205 - val_los

In [ ]:
print("AraBERT + LSTM")

model_ab_lstm, dec_emb_layer_ab_lstm, dec_dense_ab_lstm = build_seq2seq_external_emb(
    cell_type='lstm',
    encoder_emb_matrix=encoder_arabert_matrix,
    decoder_emb_matrix=decoder_arabert_matrix,
    max_enc_len=max_encoder_len,
    max_dec_len=max_decoder_len,
    embedding_dim=ARABERT_DIM,
    hidden_dim=HIDDEN_DIM,
    decoder_vocab_size=decoder_vocab_size
)
model_ab_lstm.summary()

model_ab_lstm.fit(
    [X_train_pad, decoder_input_data],
    decoder_target_data,
    batch_size=BATCH_SIZE,
    epochs=EPOCHS,
    validation_split=0.2
)

#Inference & BLEU

enc_inp_ab_lstm = model_ab_lstm.input[0]
enc_emb_ab_lstm = model_ab_lstm.layers[2].output

encoder_model_ab_lstm, decoder_model_ab_lstm = build_inference_models(
    cell_type='lstm',
    encoder_inputs=enc_inp_ab_lstm,
    enc_emb=enc_emb_ab_lstm,
    dec_emb_layer=dec_emb_layer_ab_lstm,
    decoder_dense=dec_dense_ab_lstm,
    hidden_dim=HIDDEN_DIM,
    max_dec_len=max_decoder_len,
    encoder_states_tensors=model_ab_lstm.get_layer(index=4).output
)

print("\nSample Predictions: AraBERT + LSTM")
for i in range(3):
    print("\n---------------")
    print("INPUT:", X_test.iloc[i])
    seq = X_test_pad[i].reshape(1, max_encoder_len)
    print("PREDICTED:", decode_sequence(seq, encoder_model_ab_lstm,
                                        decoder_model_ab_lstm, 'lstm', max_decoder_len))

all_scores['AraBERT_LSTM'] = evaluate_bleu(encoder_model_ab_lstm, decoder_model_ab_lstm,
                                            'lstm', 'AraBERT + LSTM')

AraBERT + LSTM


Model: "functional_38"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_55      │ (None, 150)       │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ input_layer_56      │ (None, 49)        │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_26        │ (None, 150, 768)  │  1,824,000 │ input_layer_55[0… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ not_equal_38        │ (None, 150)       │          0 │ input_layer_55[0… │
│ (NotEqual)          │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_27        │ (None, 49, 768)   │  3,135,744 │ input_layer_56[0… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm_9 (LSTM)       │ [(None, 256),     │  1,049,600 │ embedding_26[0][… │
│                     │ (None, 256),      │            │ not_equal_38[0][… │
│                     │ (None, 256)]      │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm_10 (LSTM)      │ [(None, 49, 256), │  1,049,600 │ embedding_27[0][… │
│                     │ (None, 256),      │            │ lstm_9[0][1],     │
│                     │ (None, 256)]      │            │ lstm_9[0][2]      │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_12 (Dense)    │ (None, 49, 4083)  │  1,049,331 │ lstm_10[0][0]     │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 8,108,275 (30.93 MB)

 Trainable params: 3,148,531 (12.01 MB)

 Non-trainable params: 4,959,744 (18.92 MB)

Epoch 1/8
47/47 ━━━━━━━━━━━━━━━━━━━━ 5s 55ms/step - accuracy: 0.0591 - loss: 6.8679 - val_accuracy: 0.0839 - val_loss: 6.5159
Epoch 2/8
47/47 ━━━━━━━━━━━━━━━━━━━━ 2s 46ms/step - accuracy: 0.1059 - loss: 6.3505 - val_accuracy: 0.1140 - val_loss: 6.3701
Epoch 3/8
47/47 ━━━━━━━━━━━━━━━━━━━━ 2s 50ms/step - accuracy: 0.1198 - loss: 6.1855 - val_accuracy: 0.1172 - val_loss: 6.2718
Epoch 4/8
47/47 ━━━━━━━━━━━━━━━━━━━━ 2s 46ms/step - accuracy: 0.1205 - loss: 6.1088 - val_accuracy: 0.1173 - val_loss: 6.2499
Epoch 5/8
47/47 ━━━━━━━━━━━━━━━━━━━━ 2s 46ms/step - accuracy: 0.1210 - loss: 6.0701 - val_accuracy: 0.1173 - val_loss: 6.2432
Epoch 6/8
47/47 ━━━━━━━━━━━━━━━━━━━━ 2s 46ms/step - accuracy: 0.1210 - loss: 6.0426 - val_accuracy: 0.1176 - val_loss: 6.2334
Epoch 7/8
47/47 ━━━━━━━━━━━━━━━━━━━━ 2s 47ms/step - accuracy: 0.1216 - loss: 6.0201 - val_accuracy: 0.1187 - val_loss: 6.2236
Epoch 8/8
47/47 ━━━━━━━━━━━━━━━━━━━━ 3s 54ms/step - accuracy: 0.1220 - loss: 5.9937 - val_accuracy: 0.1184 - val_loss:

In [ ]:
print("AraBERT + GRU")

model_ab_gru, dec_emb_layer_ab_gru, dec_dense_ab_gru = build_seq2seq_external_emb(
    cell_type='gru',
    encoder_emb_matrix=encoder_arabert_matrix,
    decoder_emb_matrix=decoder_arabert_matrix,
    max_enc_len=max_encoder_len,
    max_dec_len=max_decoder_len,
    embedding_dim=ARABERT_DIM,
    hidden_dim=HIDDEN_DIM,
    decoder_vocab_size=decoder_vocab_size
)
model_ab_gru.summary()

model_ab_gru.fit(
    [X_train_pad, decoder_input_data],
    decoder_target_data,
    batch_size=BATCH_SIZE,
    epochs=EPOCHS,
    validation_split=0.2
)

#Inference & BLEU

enc_inp_ab_gru = model_ab_gru.input[0]
enc_emb_ab_gru = model_ab_gru.layers[2].output

encoder_model_ab_gru, decoder_model_ab_gru = build_inference_models(
    cell_type='gru',
    encoder_inputs=enc_inp_ab_gru,
    enc_emb=enc_emb_ab_gru,
    dec_emb_layer=dec_emb_layer_ab_gru,
    decoder_dense=dec_dense_ab_gru,
    hidden_dim=HIDDEN_DIM,
    max_dec_len=max_decoder_len,
    encoder_states_tensors=model_ab_gru.get_layer(index=4).output
)

print("\nSample: AraBERT + GRU")
for i in range(3):
    print("\n-------------")
    print("INPUT:", X_test.iloc[i])
    seq = X_test_pad[i].reshape(1, max_encoder_len)
    print("PREDICTED:", decode_sequence(seq, encoder_model_ab_gru,
                                        decoder_model_ab_gru, 'gru', max_decoder_len))

all_scores['AraBERT_GRU'] = evaluate_bleu(encoder_model_ab_gru, decoder_model_ab_gru,
                                           'gru', 'AraBERT + GRU')


AraBERT + GRU


Model: "functional_41"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_60      │ (None, 150)       │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ input_layer_61      │ (None, 49)        │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_28        │ (None, 150, 768)  │  1,824,000 │ input_layer_60[0… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ not_equal_41        │ (None, 150)       │          0 │ input_layer_60[0… │
│ (NotEqual)          │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_29        │ (None, 49, 768)   │  3,135,744 │ input_layer_61[0… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ gru_9 (GRU)         │ [(None, 256),     │    787,968 │ embedding_28[0][… │
│                     │ (None, 256)]      │            │ not_equal_41[0][… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ gru_10 (GRU)        │ [(None, 49, 256), │    787,968 │ embedding_29[0][… │
│                     │ (None, 256)]      │            │ gru_9[0][1]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_13 (Dense)    │ (None, 49, 4083)  │  1,049,331 │ gru_10[0][0]      │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 7,585,011 (28.93 MB)

 Trainable params: 2,625,267 (10.01 MB)

 Non-trainable params: 4,959,744 (18.92 MB)

Epoch 1/8
47/47 ━━━━━━━━━━━━━━━━━━━━ 4s 55ms/step - accuracy: 0.0798 - loss: 6.8726 - val_accuracy: 0.1067 - val_loss: 6.4842
Epoch 2/8
47/47 ━━━━━━━━━━━━━━━━━━━━ 2s 48ms/step - accuracy: 0.1205 - loss: 6.2938 - val_accuracy: 0.1178 - val_loss: 6.3417
Epoch 3/8
47/47 ━━━━━━━━━━━━━━━━━━━━ 2s 47ms/step - accuracy: 0.1212 - loss: 6.1793 - val_accuracy: 0.1179 - val_loss: 6.3178
Epoch 4/8
47/47 ━━━━━━━━━━━━━━━━━━━━ 2s 44ms/step - accuracy: 0.1210 - loss: 6.1385 - val_accuracy: 0.1173 - val_loss: 6.3037
Epoch 5/8
47/47 ━━━━━━━━━━━━━━━━━━━━ 2s 45ms/step - accuracy: 0.1207 - loss: 6.1034 - val_accuracy: 0.1188 - val_loss: 6.2875
Epoch 6/8
47/47 ━━━━━━━━━━━━━━━━━━━━ 2s 45ms/step - accuracy: 0.1221 - loss: 6.0587 - val_accuracy: 0.1196 - val_loss: 6.2618
Epoch 7/8
47/47 ━━━━━━━━━━━━━━━━━━━━ 2s 44ms/step - accuracy: 0.1234 - loss: 6.0063 - val_accuracy: 0.1198 - val_loss: 6.2225
Epoch 8/8
47/47 ━━━━━━━━━━━━━━━━━━━━ 2s 49ms/step - accuracy: 0.1245 - loss: 5.9346 - val_accuracy: 0.1224 - val_loss:

# **Comparison**

In [ ]:
print("BLEU Score Comparison")
print("="*50)

scores_df = pd.DataFrame(
    list(all_scores.items()),
    columns=['Model', 'BLEU Score']
).sort_values('BLEU Score', ascending=False).reset_index(drop=True)

print(scores_df.to_string(index=False))

print("\nBest model:", scores_df.iloc[0]['Model'],
      "| BLEU:", round(scores_df.iloc[0]['BLEU Score'], 4))

BLEU Score Comparison
        Model  BLEU Score
   OneHot_GRU    0.000261
  OneHot_LSTM    0.000255
   OneHot_RNN    0.000132
 FastText_RNN    0.000000
FastText_LSTM    0.000000
 FastText_GRU    0.000000
       E5_RNN    0.000000
      E5_LSTM    0.000000
       E5_GRU    0.000000
  AraBERT_RNN    0.000000
 AraBERT_LSTM    0.000000
  AraBERT_GRU    0.000000

Best model: OneHot_GRU | BLEU: 0.0003
